In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import random
from pytorch_lightning.loggers import TensorBoardLogger
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, precision_score
import json
import sys
sys.path.append('..')

import os
import torch
from torch.utils.data import TensorDataset, DataLoader
import torchvision.transforms as T
import warnings
import pickle
warnings.filterwarnings("ignore")

from src.data_assemble.wrap_data import extract_splitted_data, train_val_test_split
from src.models.WindCNN import *
import copy
# %load_ext tensorboard
# %tensorboard --logdir lightning_logs/
torch.manual_seed(112)
random.seed(112)

In [2]:
path_to_data = os.path.join('..', 'data_mounted', 'nn_train')
st_split_dict = train_val_test_split(path_to_data, train = 0.5, val = 0.5, test = 0, verbose = False)
st_split_dict['Val'].append(*st_split_dict['Test'])
st_split_dict['Test'] = copy.deepcopy(st_split_dict['Val'])
print(st_split_dict)
path_to_dump = os.path.join('..', 'data_mounted', 'nn_train')


{'Train': ['Ремонтное', 'Валуйки', 'Приморско-Ахтарск', 'test', 'Богородицкое-Фенино', 'Анапа', 'Красная Поляна', 'Сочи'], 'Val': ['Гигант', 'Армавир', 'Цимлянск(Волгодонск)', 'Чертково', 'Готня', 'train', 'Краснодар, Круглик', 'Таганрог', 'Туапсе'], 'Test': ['Гигант', 'Армавир', 'Цимлянск(Волгодонск)', 'Чертково', 'Готня', 'train', 'Краснодар, Круглик', 'Таганрог', 'Туапсе']}


In [3]:
st_split_dict

{'Train': ['Ремонтное',
  'Валуйки',
  'Приморско-Ахтарск',
  'test',
  'Богородицкое-Фенино',
  'Анапа',
  'Красная Поляна',
  'Сочи'],
 'Val': ['Гигант',
  'Армавир',
  'Цимлянск(Волгодонск)',
  'Чертково',
  'Готня',
  'train',
  'Краснодар, Круглик',
  'Таганрог',
  'Туапсе'],
 'Test': ['Гигант',
  'Армавир',
  'Цимлянск(Волгодонск)',
  'Чертково',
  'Готня',
  'train',
  'Краснодар, Круглик',
  'Таганрог',
  'Туапсе']}

In [ ]:
X, y = extract_splitted_data(path_to_dump, st_split_dict)

In [ ]:
# a_file = open("../conf/conv_config.json", "r")
# json_object = json.load(a_file)
# a_file.close()
# print(json_object)

# json_object["in_channels"] = X['Train'].shape[1] # May be Edited

# a_file = open("../conf/conv_config.json", "w")
# json.dump(json_object, a_file)
# a_file.close()

In [ ]:
batch_size = 1024
with open(os.path.join('..', 'conf', 'conv_config.json')) as fs:
    args = json.load(fs)

In [ ]:
logger = TensorBoardLogger(save_dir='../logs/wind', name='windnet')
early_stop_callback = pl.callbacks.EarlyStopping(monitor="val_loss", min_delta=0.001, patience=5, verbose=False, mode="min")
trainer = pl.Trainer(max_epochs=5,
                    gpus=[0],
                    benchmark=True,
                    check_val_every_n_epoch=1,
                    # callbacks=[early_stop_callback]
)

dm = WindDataModule(X=X, y=y, batch_size=batch_size, downsample=False)
model = WindNetPL(args)

In [ ]:
trainer.fit(model, dm)

In [ ]:
trainer.test(model, dm)